In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split

# Load dataset from Google Sheets
sheet_url = 'https://docs.google.com/spreadsheets/d/1z5U1j28H1VoMoKVaKRyO7UoNZAAC9sv0E2wQem_s2mM/edit?usp=sharing'
csv_export_url = sheet_url.replace('/edit#gid=', '/export?format=csv&gid=')
data = pd.read_csv(csv_export_url)

# Preview the dataset
print(data.head())

# Data Cleaning Function
def clean_text(text):
    # Remove special characters and digits
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub('"','', text)
    text = re.sub(r"'s\b","",text)
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = ' '.join(text.split())
    text = text.lower()
    return text

# Apply the cleaning function to the article content and summaries
data['cleaned_article'] = data['Content in detail of News article'].apply(clean_text)
data['cleaned_summary'] = data['Human Summary For Article (Use Bard/ChatGpt)'].apply(clean_text)

# Drop rows with empty articles or summaries
data.dropna(subset=['cleaned_article', 'cleaned_summary'], inplace=True)

# Save the cleaned data to a CSV file
data.to_csv('cleaned_news_dataset.csv', index=False)

# Split the dataset into training, validation, and test sets
train, temp = train_test_split(data, test_size=0.2, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)

# Save the split datasets
train.to_csv('train_news_dataset.csv', index=False)
val.to_csv('val_news_dataset.csv', index=False)
test.to_csv('test_news_dataset.csv', index=False)

print("Data preprocessing complete.")


ParserError: Error tokenizing data. C error: Expected 4 fields in line 5, saw 676


In [4]:
!pip install gspread pandas


In [5]:
from google.colab import auth
auth.authenticate_user()



In [6]:
import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

# Open the Google Spreadsheet by URL
spreadsheet_url = 'https://docs.google.com/spreadsheets/d/1z5U1j28H1VoMoKVaKRyO7UoNZAAC9sv0E2wQem_s2mM/edit?usp=sharing'
spreadsheet = gc.open_by_url(spreadsheet_url)

# Select the first sheet
worksheet = spreadsheet.sheet1


In [7]:
import pandas as pd

# Get all values in the sheet
data = worksheet.get_all_values()

# Convert to pandas DataFrame
df = pd.DataFrame(data[1:], columns=data[0])


In [9]:
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

# Function to clean text
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    text = re.sub(r'\[.*?\]', '', text)  # Remove text in square brackets
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # Remove URLs
    text = re.sub(r'<.*?>+', '', text)  # Remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove non-alphanumeric characters
    text = text.lower()  # Convert to lowercase
    text = ' '.join(word for word in text.split() if word not in stopwords.words('english'))  # Remove stopwords
    return text

# Apply cleaning function to the relevant columns
df['Content in detail of News article'] = df['Content in detail of News article'].apply(clean_text)
df['Human Summary For Article (Use Bard/ChatGpt or Any other tool for summarization)'] = df['Human Summary For Article (Use Bard/ChatGpt or Any other tool for summarization)'].apply(clean_text)

# Handle missing values (if any)
df.dropna(inplace=True)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


KeyError: 'Content in detail of News article'

In [10]:
# Print column names to verify
print(df.columns)


Index(['Timestamp', 'Please enter your full name',
       'Please write the Name of Newspaper - Newspaper Must be Indian English Newspaper listed above',
       'Published date of News\nDate when the news article  was published.',
       'Headline of News Article\nHeadline / Title of the News Article from Source ',
       'Content in detail of News article\nArticle in detail from data source (Make sure, data must be very clear)',
       'Human Summary For Article (Use Bard/ChatGpt or Any other tool for summarization)\nWrite a summary of the article. Explain the news in your own words. Minimum 50 words and maximum 200 words',
       'News Category (Please enter correctly)',
       'Enter URL or Link of News \n(Eg: - Modi Cabinet 2024: Full list of Cabinet Ministers in Narendra Modi Government 3.0)'],
      dtype='object')


In [11]:
# Step 1: Install the required libraries
!pip install gspread pandas

# Step 2: Authorize access to Google Sheets
from google.colab import auth
auth.authenticate_user()

# Step 3: Access the Google Spreadsheet
import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

spreadsheet_url = 'https://docs.google.com/spreadsheets/d/1z5U1j28H1VoMoKVaKRyO7UoNZAAC9sv0E2wQem_s2mM/edit?usp=sharing'
spreadsheet = gc.open_by_url(spreadsheet_url)
worksheet = spreadsheet.sheet1

# Step 4: Load the data into a pandas DataFrame
import pandas as pd

data = worksheet.get_all_values()
df = pd.DataFrame(data[1:], columns=data[0])

# Print column names to verify
print(df.columns)

# Step 5: Preprocess the data
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()
    text = ' '.join(word for word in text.split() if word not in stopwords.words('english'))
    return text

# Correct column names with newlines and additional text
content_column = 'Content in detail of News article\nArticle in detail from data source (Make sure, data must be very clear)'
summary_column = 'Human Summary For Article (Use Bard/ChatGpt or Any other tool for summarization)\nWrite a summary of the article. Explain the news in your own words. Minimum 50 words and maximum 200 words'

# Use the correct column names for preprocessing
df[content_column] = df[content_column].apply(clean_text)
df[summary_column] = df[summary_column].apply(clean_text)

df.dropna(inplace=True)

# Step 6: Save the preprocessed data
df.to_csv('preprocessed_news_data.csv', index=False)


Index(['Timestamp', 'Please enter your full name',
       'Please write the Name of Newspaper - Newspaper Must be Indian English Newspaper listed above',
       'Published date of News\nDate when the news article  was published.',
       'Headline of News Article\nHeadline / Title of the News Article from Source ',
       'Content in detail of News article\nArticle in detail from data source (Make sure, data must be very clear)',
       'Human Summary For Article (Use Bard/ChatGpt or Any other tool for summarization)\nWrite a summary of the article. Explain the news in your own words. Minimum 50 words and maximum 200 words',
       'News Category (Please enter correctly)',
       'Enter URL or Link of News \n(Eg: - Modi Cabinet 2024: Full list of Cabinet Ministers in Narendra Modi Government 3.0)'],
      dtype='object')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
import pandas as pd

# Load the preprocessed data
df = pd.read_csv('preprocessed_news_data.csv')
print(df.head())


           Timestamp Please enter your full name  \
0   5/1/2024 8:14:22                 Anish Kumar   
1   5/1/2024 9:13:57                 Anish Kumar   
2   5/1/2024 9:21:18                 Anish Kumar   
3   5/1/2024 9:24:29                 Anish Kumar   
4  5/1/2024 19:07:21          Puneet Kumar Gupta   

  Please write the Name of Newspaper - Newspaper Must be Indian English Newspaper listed above  \
0                                 The Assam Tribune                                              
1                                 The Assam Tribune                                              
2                                 The Assam Tribune                                              
3                                 The Assam Tribune                                              
4                                        The Pioneer                                             

  Published date of News\nDate when the news article  was published.  \
0                         

In [13]:
!pip install transformers

from transformers import PegasusTokenizer, PegasusForConditionalGeneration

# Load the Pegasus tokenizer and model
model_name = 'google/pegasus-xsum'
tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.52M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

In [ ]:
def summarize_text(text):
    tokens = tokenizer(text, truncation=True, padding='longest', return_tensors="pt")
    summary_ids = model.generate(tokens['input_ids'])
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# Apply summarization to the news content
df['Generated Summary'] = df[content_column].apply(summarize_text)
print(df[['Generated Summary']].head())
